In [ ]:
import numpy as np
import pandas as pd

# replace with your change-metrics list (or keep the one below)
change_metrics = [
    'Residential_Electricity_Share_Change',
    'Residential_Electricity_Level_Change',
    'Residential_Polluting_Level_Change',
    'CSMI_Electricity_Share_Change',
    'CSMI_Electricity_Level_Change',
    'CSMI_Polluting_Level_Change',
    'Transport_Low_Emission_Share_Change',
    'Transport_Low_Emission_Level_Change',
    'Transport_Polluting_Level_Change',
    'CSMI_Total_Energy_Share_Change',
    
    
]

# filter to the requested change metrics
df = tidy_all_data[tidy_all_data['Metric'].isin(change_metrics)].copy()
df['Value'] = pd.to_numeric(df['Value'], errors='coerce')

# drop infinite values and keep NaNs to be explicit
df['Value'].replace([np.inf, -np.inf], np.nan, inplace=True)

# compute per-metric mean and std (skip metrics with insufficient data)
grp = df.groupby('Metric')['Value']
df['mean'] = grp.transform('mean')
df['std'] = grp.transform('std')
df['median'] = grp.transform('median')

# avoid division by zero: mark std==0 as non-outlier
df['zscore'] = (df['Value'] - df['mean']) / df['std']
df.loc[df['std'] == 0, 'zscore'] = np.nan

# flag outliers: absolute z-score > 3
df['is_outlier_3sd'] = df['zscore'].abs() > 3

# produce summary of results
summary = df.groupby('Metric').agg(
    n_total=('Value','size'),
    n_nonnull=('Value','count'),
    n_outliers=('is_outlier_3sd','sum'),
    mean=('mean','first'),
    median=('median','first'),
    std=('std','first')
).reset_index()

print("Outlier summary per metric:")
display(summary)

# list of outlier rows (municipality, year, value, zscore)
outliers = df[df['is_outlier_3sd']].sort_values(['Metric','zscore'], ascending=[True, False])
print(f"\nTotal outlier rows found: {len(outliers)}")
display(outliers[['Municipality_Name','Year','Metric','Value','zscore']].reset_index(drop=True))
